# GRPO-train Qwen2.5-1.5B on `exploitable_poker`

End-to-end on a Colab A100 (or H100). Two modes are available:

- **Plain GRPO** — one CLBench instance per rollout, no memory.
- **Notepad mode** — four instances per rollout with `icl_notepad`-style memory; the policy learns to write durable notes that boost reward on later instances.

See the project README for the full architecture diagram and rationale.

## 1. Clone and bootstrap

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch the runtime to A100/H100 first.'
%cd /content
![ -d clbench-verifiers ] || git clone https://github.com/sr-networks/clbench-verifiers.git
%cd clbench-verifiers
!bash scripts/colab_setup.sh

## 2. CPU smoke test (no GPU work)

Build the env, drive a few turns, confirm everything wires up before burning GPU time.

In [ ]:
!python tests/test_env_smoke.py

## 3. Pick a config and train

Run *one* of the two cells below. They write to different `output_dir`s so they don't conflict if you decide to do both.

In [ ]:
# 3a. Plain GRPO (single-instance rollouts, no memory)
!clbv-train --config configs/poker_qwen2_5_1_5b.toml

In [ ]:
# 3b. Notepad GRPO (4-instance rollouts with icl_notepad-style memory)
!clbv-train --config configs/poker_qwen2_5_1_5b_notepad.toml

## 4. Evaluate via the official CLBench harness

`clbv-eval` spawns vLLM serving the checkpoint and shells out to `clbench run` against `--system vllm_local`. For notepad-trained models, pass `--clbench-arg=--system.use_notepad=true` to keep the eval-time interface aligned with what was trained.

In [ ]:
# Evaluate plain checkpoint (use the path from cell 3a)
!clbv-eval --checkpoint ./checkpoints/poker_qwen2_5_1_5b/final \
    --task exploitable_poker --schedule quick_test

In [ ]:
# Evaluate notepad checkpoint (use the path from cell 3b)
!clbv-eval --checkpoint ./checkpoints/poker_qwen2_5_1_5b_notepad/final \
    --task exploitable_poker --schedule quick_test \
    --clbench-arg=--system.use_notepad=true

## 5. (Optional) Compare against CLBench's published baseline

The `final_results/` dir in the cloned `continual-learning-bench` repo has runs for ICL/ACE/mem0 etc. on the same task. The reward you should beat is the `reset_each_instance` baseline shown in the per-task `*.json.gz` summary.

In [ ]:
import gzip, json
from pathlib import Path
p = Path('continual-learning-bench/final_results/runs/icl-gpt-5.4/tasks/exploitable_poker.json.gz')
if p.exists():
    with gzip.open(p) as f:
        d = json.load(f)
    print('icl-gpt-5.4 mean reward:', d['summary']['aggregate']['score']['mean'])
    print('reset_each_instance baseline mean reward:', d['baseline_trace']['result']['score'])
else:
    print('Skip — clone the bench at expected path or update p.')